# 지시문 · 모델 실험 (image-classification)

DEV case 몇 개에 지시문과 모델을 시험하고, 원시 답을 본 뒤 Go로 내보낸다.

- 공급자 key는 환경의 `ANTHROPIC_API_KEY` / `OPENAI_API_KEY`에서 온다. 어디에도 쓰지 않는다.
- `RUN = False`면 아무것도 부르지 않고 모든 case를 미호출로 기록하므로 notebook 나머지가 그대로 돈다.

In [ ]:
from pathlib import Path

from snapdone_eval_lab import analysis
from snapdone_eval_lab.datasets import load_dataset, repository_root
from snapdone_eval_lab.experiments import choose_cases, export_predictions, export_replay_variant, load_prompt, replay_command, run_classification_experiment
from snapdone_eval_lab.plots import bar, show_image

ROOT = repository_root()

# ---- 실험 변수 -----------------------------------------------------------
EXPERIMENT_ID = "lab-facts-prompt"      # 소문자 · 숫자 · dash: variantId와 파일 이름이 된다
DATASET = "sample-classification"
PROVIDER = "anthropic"                   # "anthropic" 또는 "openai" (OpenAI · Ollama · vLLM)
MODEL = "claude-opus-5"                  # 공급자가 제공하는 아무 모델 id
BASE_URL = None                          # openai만. 예: "http://localhost:11434/v1"
TEMPERATURE = None                       # None이면 production처럼 공급자 기본값을 보낸다
PROMPT = load_prompt(ROOT / "tools/evals/experiments/prompts/facts-normalized.md")  # 또는 문자열을 직접
CASE_LIMIT = 5                           # 실행당 유료 호출 수. run_classification_experiment는 max_calls를 넘으면 거절한다
RUN = False                              # True면 실제로 호출한다
# ---------------------------------------------------------------------------

print(PROMPT)

In [ ]:
dataset = load_dataset(DATASET, split="dev")
cases = choose_cases(dataset, limit=CASE_LIMIT)  # ids=[...]나 category="event"를 더해 좁힌다
dataset.frame[dataset.frame["id"].isin(cases)][["id", "category", "intent", "acceptable_actions", "fact_count"]]

In [ ]:
experiment = run_classification_experiment(
    dataset,
    cases,
    experiment_id=EXPERIMENT_ID,
    provider=PROVIDER,
    model=MODEL,
    prompt=PROMPT,
    temperature=TEMPERATURE,
    base_url=BASE_URL,
    max_calls=CASE_LIMIT,
    dry_run=not RUN,
)
frame = analysis.results_frame(experiment, dataset)
frame[["case_id", "gold_category", "predicted_category", "predicted_action", "confidence", "usable", "problem", "elapsed_ms"]]

In [ ]:
# 원시 답: 모델이 실제로 돌려준 것, case마다 하나
for r in experiment.results:
    print(f"--- {r.case_id} · {r.raw.answered_model} · {r.raw.stop_reason} · {r.raw.input_tokens}/{r.raw.output_tokens} 토큰")
    print(r.raw.text or f"(텍스트 없음: {r.problem})")

In [ ]:
# 예측 분포 · 혼동표 · category별 일치율
display(analysis.distribution(frame, "predicted_category"))
display(analysis.distribution(frame, "confidence"))
display(analysis.confusion_table(frame))
print("연구용 category 일치율:", analysis.research_category_match_rate(frame))
analysis.slice_by(frame, "gold_category")

In [ ]:
# 하나씩 살펴볼 실패와 high confidence 답
display(analysis.failures(frame))
analysis.high_confidence(frame)[["case_id", "gold_category", "predicted_category", "predicted_action"]]

In [ ]:
# case 하나를 그 답과 나란히 본다
CASE = cases[0]
show_image(dataset, CASE)
frame[frame["case_id"] == CASE].T

In [ ]:
# 원시 예측을 Go로 내보낸다. 임시 결과는 lab/out(git 제외), 남길 것은 tools/evals/predictions로 옮긴다.
OUT = ROOT / "tools/evals/lab/out"
predictions = export_predictions(experiment, OUT / f"{EXPERIMENT_ID}.jsonl")
variant = export_replay_variant(experiment, OUT / f"{EXPERIMENT_ID}.json", ROOT / "tools/evals/variants/replay-example.json")
print(predictions.read_text(encoding="utf-8"))
print("저장소 루트에서 실행:")
print(replay_command(dataset, predictions, variant, ROOT, run_id=EXPERIMENT_ID))